<img src="banner.png" width="100%" style="max-height:300px; object-fit:cover;"/>

# Curso de Agentes de IA con LangGraph

Este notebook sirve como una guía introductoria para el diseño e implementación de **Agentes de Inteligencia Artificial** utilizando el framework **LangChain** y la integración de la familia **Gemini** a través de **Google Cloud Vertex AI**.

A lo largo de este curso, aprenderemos a:

- Conectar modelos de lenguaje con herramientas externas.
- Crear flujos de trabajo cíclicos basados en grafos.
- Dotar a los sistemas de capacidades de razonamiento.
- Diseñar agentes con toma de decisiones autónoma.

> El objetivo es construir una base sólida para desarrollar agentes inteligentes capaces de interactuar con herramientas, procesar información y ejecutar tareas de forma dinámica.


## Paso 1: Instalación de Dependencias

Para comenzar, instalaremos todas las librerías necesarias para el curso en su versión más reciente. Esto incluye:

- **langchain**, **langchain-community**, **langchain-google-genai**: módulos principales y de integración con Google Generative AI (Gemini).
- **langgraph**: motor para crear agentes basados en grafos con estados y ciclos.
- **python-dotenv**: carga de variables de entorno desde un archivo `.env`.
- **arxiv**: herramienta de búsqueda y consulta académica para ejercicios del curso.

> **Nota:** usamos el parámetro `-q` (*quiet*) para realizar una instalación limpia y libre de barras de progreso ruidosas.


In [14]:
# Instalar dependencias del curso de forma silenciosa
!pip install -q -U langchain langchain-community langgraph langchain-google-genai langchain-tavily google-auth python-dotenv arxiv tavily-python

## Paso 2: Autenticación con Google Cloud (ADC)

Para conectarnos a **Vertex AI** sin usar una API key, utilizamos **Application Default Credentials (ADC)**. Este mecanismo detecta automáticamente las credenciales del entorno local.

Ejecuta los siguientes comandos **una sola vez** desde la terminal integrada de DataSpell (`Alt + F12`):

```bash
# Iniciar sesión con tu cuenta de Google
gcloud auth login

# Generar el archivo de credenciales ADC en tu entorno local
gcloud auth application-default login

# Establecer tu proyecto de Google Cloud (debe tener Vertex AI habilitado)
gcloud config set project TU_PROJECT_ID
```

Las credenciales se almacenan en `%APPDATA%\gcloud\application_default_credentials.json` y son detectadas automáticamente por el SDK.

> **Nota:** el `project_id` se carga desde un archivo `.env` para no hardcodearlo en el código.


## Paso 3: Configuración del Entorno e Importación de Librerías

En este paso, configuraremos el entorno de Python e importaremos las librerías esenciales para trabajar con **LangChain** y **LangGraph**.

Esto incluye:

- **warnings**: módulo utilizado para controlar y filtrar advertencias durante la ejecución del notebook.
- **dotenv**: carga del `GOOGLE_CLOUD_PROJECT` y `TAVILY_API_KEY` desde el archivo `.env`.
- **google.auth**: obtención de credenciales ADC para autenticación con Google Cloud sin API key.
- **langchain-google-genai**: integración de LangChain con los modelos Gemini de Google.
- **LangGraph**: extensión que permite crear flujos de trabajo basados en grafos, estados y ciclos.

> **Nota:** configuraremos un filtro de advertencias para mantener la salida del notebook limpia, evitando mensajes de depreciación o avisos menores que no afecten el desarrollo del curso.


In [15]:
import os
import warnings
from dotenv import load_dotenv

load_dotenv()

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
import langgraph

print("✅ Entorno configurado e importaciones listas.")

✅ Entorno configurado e importaciones listas.


## Paso 4: Inicialización del Modelo, Herramientas y Prueba de Conexión

En este paso, inicializaremos el modelo **Gemini 2.5 Flash**, definiremos las herramientas del agente y realizaremos una prueba de conexión.

Esto incluye:

- **Inicialización del modelo**: creación de una instancia de `ChatGoogleGenerativeAI` autenticada mediante ADC con Vertex AI.
- **busca_web**: herramienta que realiza búsquedas en la web usando Tavily Search.
- **multiplicar**: herramienta que multiplica dos números enteros.
- **bind_tools**: vinculación de las herramientas al modelo para que pueda invocarlas.
- **Prueba de conexión**: verificación de que las credenciales ADC y la configuración del entorno están funcionando.

> **Nota:** esta prueba permite confirmar que el entorno está listo para comenzar a trabajar con modelos de lenguaje dentro de flujos creados con LangChain y LangGraph.


In [16]:
import google.auth
from langchain_google_genai import ChatGoogleGenerativeAI

# Obtener credenciales ADC y project_id
credentials, project_id = google.auth.default()

# Inicializar el modelo Gemini 2.5 Flash con credenciales ADC
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    credentials=credentials
)

@tool
def busca_web(query: str) -> list:
    """Realiza una búsqueda en la web sobre un tema específico"""
    tavily_search = TavilySearch(max_results=5)
    return tavily_search.invoke(query)

@tool
def multiplicar(a: int, b: int) -> int:
    """Multiplica dos números enteros y devuelve el resultado. Use esta herramienta para cualquier operación de multiplicación."""
    return a * b

tools = [busca_web, multiplicar]
llm_con_herramientas = llm.bind_tools(tools)

print("\n🧰 Herramientas registradas:")
for t in tools:
    print(f"  ✅ {t.name:<15} — {t.description}")

mensaje = HumanMessage(content="¡Hola! ¿Estás inicializado correctamente?")

try:
    respuesta = llm.invoke([mensaje])
    print("✅ Conexión con Vertex AI establecida de forma exitosa.")
    print("Respuesta de Gemini:")
    print(respuesta.content)
except Exception as e:
    print("❌ Error de conexión detectado!")
    print(e)


🧰 Herramientas registradas:
  ✅ busca_web       — Realiza una búsqueda en la web sobre un tema específico
  ✅ multiplicar     — Multiplica dos números enteros y devuelve el resultado. Use esta herramienta para cualquier operación de multiplicación.
✅ Conexión con Vertex AI establecida de forma exitosa.
Respuesta de Gemini:
¡Hola! Sí, estoy inicializado y listo para ayudarte. ¿En qué puedo asistirte hoy?


## Paso 5: Prueba de Herramientas

En este paso probaremos que las herramientas vinculadas al modelo funcionan correctamente de forma independiente.

Esto incluye:

- **busca_web**: el modelo detecta que debe buscar en la web y retorna resultados reales de Tavily.
- **multiplicar**: el modelo detecta la operación matemática y devuelve el resultado exacto.

> **Nota:** en este paso invocamos las herramientas manualmente a partir de los `tool_calls` que genera el modelo, simulando lo que haría un agente de forma autónoma.


In [17]:
import re, html

def limpiar_texto(texto: str) -> str:
    texto = html.unescape(texto)                        # &amp; → & , &#39; → '
    texto = re.sub(r'\[.*?\]\(.*?\)', '', texto)   # elimina links markdown
    texto = re.sub(r'\s+', ' ', texto).strip()        # colapsa espacios/saltos
    return texto

# Prueba con búsqueda web
mensaje_busqueda = HumanMessage(content="¿Cuáles son las últimas noticias sobre LangGraph?")
respuesta_busqueda = llm_con_herramientas.invoke([mensaje_busqueda])
resultado_busqueda = busca_web.invoke(respuesta_busqueda.tool_calls[0]["args"])

resultados = resultado_busqueda.get('results', []) if isinstance(resultado_busqueda, dict) else resultado_busqueda
print(f"\n📰 Resultados de Tavily ({len(resultados)} encontrados)\n" + "─" * 60)
for i, r in enumerate(resultados, 1):
    titulo  = limpiar_texto(r.get('title', 'Sin título'))
    url     = r.get('url', '')
    snippet = limpiar_texto(r.get('content', ''))[:300]
    print(f"\n{i}. 📌 {titulo}")
    print(f"   🔗 {url}")
    print(f"   {snippet}...")
    print("   " + "─" * 57)

# Prueba con multiplicación
mensaje_multiplicar = HumanMessage(content="¿Cuánto es 17 multiplicado por 43?")
respuesta_multi = llm_con_herramientas.invoke([mensaje_multiplicar])
resultado_multi = multiplicar.invoke(respuesta_multi.tool_calls[0]["args"])
print(f"\n🔢 Resultado de multiplicación: {resultado_multi}")


📰 Resultados de Tavily (5 encontrados)
────────────────────────────────────────────────────────────

1. 📌 LangGraph Flaw Chain Exposes Self-Hosted AI Agents to Remote ...
   🔗 https://thehackernews.com/2026/06/langgraph-flaw-chain-exposes-self.html
   # LangGraph Flaw Chain Exposes Self-Hosted AI Agents to Remote Code Execution. Cybersecurity researchers have disclosed details of three now-patched security flaws impacting LangGraph, including a critical vulnerability chain that could result in remote code execution. "An SQL injection in LangGraph...
   ─────────────────────────────────────────────────────────

2. 📌 LangGraph Engineer - Hacker News
   🔗 https://news.ycombinator.com/item?id=41203307
   | | | | --- | | LangGraph Engineer (github.com/hwchase17) | | 63 points by gfortaine on Aug 9, 2024 | hide | past | favorite | 38 comments | | | | | --- | --- | | | | | --- | | dbmikus on Aug 9, 2024 | next ) I was thinking through, what are the benefits of coding up your agent system via